### Setup

In [3]:
import sagemaker
import boto3
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.s3 import S3Uploader

sagemaker_session = sagemaker.Session()
pipeline_session = PipelineSession()

region = boto3.Session().region_name
role = sagemaker.get_execution_role()

default_bucket = sagemaker_session.default_bucket()

bucket = sagemaker.Session().default_bucket()

In [4]:
input_data_uri = S3Uploader.upload(
    local_path="../data/raw",
    desired_s3_uri=f"s3://{default_bucket}/pipeline-raw"
)

print(input_data_uri)

s3://sagemaker-us-east-1-110276528929/pipeline-raw


### Parámetros

In [5]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1
)

instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large"
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

input_data = ParameterString(
    name="InputData",
    default_value=input_data_uri
)

batch_data = ParameterString(
    name="BatchData",
    default_value=f"s3://{default_bucket}/batch-data"
)

rmse_threshold = ParameterFloat(
    name="RmseThreshold",
    default_value=1.0
)

## ProcessingStep

### Processor

In [6]:
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1
)

In [7]:
from sagemaker.processing import ScriptProcessor

processing_image_uri = "110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing-byoc:latest"

processor = ScriptProcessor(
    image_uri=processing_image_uri,
    command=["python"],
    instance_type="ml.m5.large",
    instance_count=processing_instance_count,
    base_job_name="prep-byoc",
    role=role,
    sagemaker_session=pipeline_session,
)

### Step args 

In [8]:
from sagemaker.processing import ProcessingInput, ProcessingOutput

processor_args = processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test"
        ),
    ],
    code="../src/preprocessing/prep.py",
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


### Step

In [9]:
from sagemaker.workflow.steps import ProcessingStep

step_process = ProcessingStep(
    name="ProcessData",
    step_args=processor_args
)

## TrainingStep

### Estimator

In [20]:
from sagemaker.workflow.parameters import ParameterInteger

training_instance_count = ParameterInteger(
    name="TrainingInstanceCount",
    default_value=1
)

In [21]:
from sagemaker.estimator import Estimator

training_image_uri = "110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-training-byoc:latest"

estimator = Estimator(
    image_uri=training_image_uri,
    role=role,
    instance_count=training_instance_count,
    instance_type="ml.m5.large",
    base_job_name="train-byoc",
    sagemaker_session=pipeline_session,
)

### Step args

In [22]:
train_args = estimator.fit(
    inputs={
        "train": step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
        "validation": step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
    }
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


### Step

In [23]:
from sagemaker.workflow.steps import TrainingStep

step_train = TrainingStep(
    name="TrainModel",
    step_args=train_args
)

## EvaluationStep

### Processor (evaluation)

In [24]:
evaluation_image_uri = "110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-evaluation-byoc:latest"

eval_processor = ScriptProcessor(
    image_uri=evaluation_image_uri,
    command=["python"],
    instance_type="ml.m5.large",
    instance_count=processing_instance_count,
    base_job_name="eval-byoc",
    role=role,
    sagemaker_session=pipeline_session,
)

### Step args

In [25]:
eval_args = eval_processor.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/input/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/input/test"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation"
        )
    ],
    code="../src/evaluation/evaluate.py",
)

### Step

In [26]:
from sagemaker.workflow.properties import PropertyFile

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

In [27]:
step_eval = ProcessingStep(
    name="EvaluateModel",
    step_args=eval_args,
    property_files=[evaluation_report],
)

## ModelStep

In [28]:
from sagemaker.model import Model

inference_image_uri = "110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-inference-byoc:v2"

model = Model(
    image_uri=inference_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
)

### Step

In [29]:
from sagemaker.workflow.model_step import ModelStep

step_model = ModelStep(
    name="CreateModel",
    step_args=model.create()
)

## TransformStep

In [30]:
from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep
from sagemaker.inputs import TransformInput

transformer = Transformer(
    model_name=step_model.properties.ModelName,
    instance_type="ml.m5.large",
    instance_count=1,
    output_path=f"s3://{bucket}/transform-output",
    sagemaker_session=pipeline_session,
)

### Step

In [31]:
step_transform = TransformStep(
    name="BatchTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
        content_type="text/csv"
    )
)

## ModelStep (registro) 

In [32]:
from sagemaker.model_metrics import ModelMetrics, MetricsSource

metrics_source = MetricsSource(
    s3_uri=step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
    content_type="application/json",
)

model_metrics = ModelMetrics(
    model_statistics=metrics_source
)

### Step

In [33]:
step_register = ModelStep(
    name="RegisterModel",
    step_args=model.register(
        content_types=["application/json"],
        response_types=["application/json"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name="ml-model-group",
        model_metrics=model_metrics,
    )
)

## ConditionStep

In [34]:
from sagemaker.workflow.conditions import ConditionLessThan
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import JsonGet

### Threshold

In [35]:
rmse_threshold = 1.0

### Condición

In [36]:
condition = ConditionLessThan(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,  
        json_path="rmse"
    ),
    right=rmse_threshold
)

## FailStep

### Fail 

In [37]:
fail_step = FailStep(
    name="FailIfPoorModel",
    error_message="Modelo no cumple el threshold de RMSE"
)

### Step

In [38]:
step_condition = ConditionStep(
    name="CheckRMSE",
    conditions=[condition],
    if_steps=[step_register, step_transform],
    else_steps=[fail_step]
)

## Pipeline

In [39]:
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name="PipelineBYOC",
    parameters=[
        processing_instance_count,
        training_instance_count,
        input_data
    ],
    steps=[
        step_process,
        step_train,
        step_eval,
        step_model,
        step_condition
    ],
    sagemaker_session=pipeline_session,
)

## Ejecución

In [40]:
pipeline.upsert(role_arn=role)

execution = pipeline.start()

execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:110276528929:pipeline/PipelineBYOC',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:110276528929:pipeline/PipelineBYOC/execution/5gar30f0pfs8',
 'PipelineExecutionDisplayName': 'execution-1774430833877',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2026, 3, 25, 9, 27, 13, 801000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 3, 25, 9, 27, 13, 801000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:110276528929:user-profile/d-if0ke2gbisqa/datascientist',
  'UserProfileName': 'datascientist',
  'DomainId': 'd-if0ke2gbisqa',
  'IamIdentity': {'Arn': 'arn:aws:sts::110276528929:assumed-role/SageMakerStudioExecutionRole2026/SageMaker',
   'PrincipalId': 'AROARTLH6JMQ7WIOJYQJB:SageMaker'}},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:110276528929:user-profile/d-if0ke2gbisqa/datascientist',
  'UserProfileName': 'datascientist',
  'DomainId

## Verificación

In [43]:
execution.wait()
execution.list_steps()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 execution.wait()                                                                             │
│   2 execution.list_steps()                                                                       │
│   3                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:938 in wait               │
│                                                                                                  │
│    935 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    936 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    937 │   │   )                                                                                 │
│ ❱  938 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    939 │                                                                                         │
│    940 │   def result(self, step_name: str):                                                     │
│    941 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:55 in wait                            │
│                                                                                                  │
│    52 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    53 │   # method.                                                                              │
│    54 │   def wait(self, **kwargs):                                                              │
│ ❱  55 │   │   Waiter.wait(self, **kwargs)                                                        │
│    56 │                                                                                          │
│    57 │   wait.__doc__ = WaiterDocstring(                                                        │
│    58 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:374 in wait                           │
│                                                                                                  │
│   371 │   │   │   │   return                                                                     │
│   372 │   │   │   if current_state == 'failure':                                                 │
│   373 │   │   │   │   reason = f'Waiter encountered a terminal failure state: {acceptor.explan   │
│ ❱ 374 │   │   │   │   raise WaiterError(                                                         │
│   375 │   │   │   │   │   name=self.name,                                                        │
│   376 │   │   │   │   │   reason=reason,                                                         │
│   377 │   │   │   │   │   last_response=response,                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
WaiterError: Waiter PipelineExecutionComplete failed: Waiter encountered a terminal failure state: For expression 
"PipelineExecutionStatus" we matched expected path: "Failed"

In [107]:
import os
os.getcwd()

'/home/sagemaker-user/Tarea3_ProductoDeDatos/notebooks'

In [44]:
execution.list_steps()

[{'StepName': 'ProcessData',
  'StartTime': datetime.datetime(2026, 3, 25, 9, 27, 14, 862000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 3, 25, 9, 29, 48, 506000, tzinfo=tzlocal()),
  'StepStatus': 'Failed',
  'FailureReason': 'ClientError: AlgorithmError: , exit code: 1',
  'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:110276528929:processing-job/pipelines-5gar30f0pfs8-ProcessData-u50EYKOYUa'}},
  'AttemptCount': 1}]